In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# !pip install scispacy
!pip install /content/drive/MyDrive/eq_5d/scispacy/en_core_sci_sm-0.5.4.tar.gz
!pip install /content/drive/MyDrive/eq_5d/scispacy/en_core_sci_md-0.5.4.tar.gz
!pip install /content/drive/MyDrive/eq_5d/scispacy/en_core_sci_scibert-0.5.4.tar.gz

In [ ]:
import os
import random
import numpy as np
import pandas as pd
from tqdm import tqdm
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report, f1_score
from sklearn.utils.class_weight import compute_class_weight

import torch
from torch import nn
from torch.utils.data import DataLoader, Dataset
from torch.optim import AdamW
from transformers import AutoTokenizer, AutoModel
import spacy

In [ ]:
DATA_CSV = "drive/MyDrive/eq_5d/eq-5d-200-records.csv"
TEXT_COL = "Abstract"
LABEL_COL = "Label"
ID_COL = "No"

ENCODER_MODELS = {
    "BERT": "bert-base-uncased",
    "SciBERT": "allenai/scibert_scivocab_uncased",
    "BioBERT": "dmis-lab/biobert-base-cased-v1.1"
}

SCISPACY_MODELS = {
    "sm": "en_core_sci_sm",
    "md": "en_core_sci_md",
    "scibert": "en_core_sci_scibert"
}

In [ ]:
LEARNING_RATES = [2e-5, 5e-6, 2e-6, 1e-6]
EPOCHS = 20
EARLY_STOP = 5
MAX_LEN = 256
BATCH_SIZE = 1
REPEATS = 5
SEED = 42
OUTPUT_DIR = "drive/MyDrive/eq_5d/experiments/"
os.makedirs(OUTPUT_DIR, exist_ok=True)

In [ ]:
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

In [ ]:
def enrich_sentence(sent_text, nlp):
    doc = nlp(sent_text)
    ents = [f"{ent.text.strip()}|{ent.label_}" for ent in doc.ents if ent.text.strip()]
    if ents:
        unique = list(dict.fromkeys(ents))[:30]
        return sent_text + " [ENTS: " + "; ".join(unique) + "]"
    return sent_text

In [ ]:
# nlp = spacy.load("en_core_sci_sm")

In [ ]:
def split_and_enrich(df, nlp, text_col, id_col, label_col):
    bags = []
    for _, row in tqdm(df.iterrows(), total=len(df)):
        text = str(row[text_col]) if pd.notna(row[text_col]) else ""
        if not text:
            continue
        doc = nlp(text)
        sents = []
        for sent in doc.sents:
            s = sent.text.strip()
            if s:
                sents.append(enrich_sentence(s, nlp))
        bags.append({
            "bag_id": row[id_col],
            "sentences": sents,
            "label": int(row[label_col])
        })
    return bags

In [ ]:
class MILDataset(Dataset):
    def __init__(self, bags, tokenizer, max_len):
        self.bags = bags
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.bags)

    def __getitem__(self, idx):
        bag = self.bags[idx]
        enc = self.tokenizer(
            bag["sentences"],
            max_length=self.max_len,
            padding="max_length",
            truncation=True,
            return_tensors="pt"
        )
        return {
            "input_ids": enc["input_ids"],
            "attention_mask": enc["attention_mask"],
            "label": torch.tensor(bag["label"], dtype=torch.long),
            "bag_id": bag["bag_id"],
            "sentences": bag["sentences"]
        }

In [ ]:
# tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

In [ ]:
class MILModel(nn.Module):
    def __init__(self, model_name, hidden_size=768, num_labels=2, freeze_encoder=True):
        super(MILModel, self).__init__()
        self.encoder = AutoModel.from_pretrained(model_name)
        if freeze_encoder:
            for param in self.encoder.parameters():
                param.requires_grad = False
        self.attention = nn.Sequential(
            nn.Linear(hidden_size, 128),
            nn.Tanh(),
            nn.Linear(128, 1)
        )
        self.classifier = nn.Linear(hidden_size, num_labels)

    def forward(self, input_ids, attention_mask, return_attn=False):
        outputs = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        sent_embeddings = outputs.last_hidden_state[:,0,:]  # CLS token
        attn_logits = self.attention(sent_embeddings)
        attn_weights = torch.softmax(attn_logits, dim=0)
        bag_rep = torch.sum(attn_weights * sent_embeddings, dim=0)
        logits = self.classifier(bag_rep)
        if return_attn:
            return logits, attn_weights.squeeze(-1)
        else:
            return logits

In [ ]:
def train_eval(train_loader, val_loader, device, class_weights, model_name, lr):
    model = MILModel(model_name, freeze_encoder=True).to(device)
    optimizer = AdamW(model.parameters(), lr=lr, eps=1e-8)
    best_f1, best_state = -1, None
    patience = 0

    for epoch in range(EPOCHS):
        model.train()
        total_loss = 0
        for batch in train_loader:
            input_ids = batch["input_ids"].squeeze(0).to(device)
            attn_mask = batch["attention_mask"].squeeze(0).to(device)
            labels = batch["label"].to(device)
            if labels.dim() == 0:
                labels = labels.unsqueeze(0)

            model.zero_grad()
            logits = model(input_ids, attn_mask).unsqueeze(0)
            loss = torch.nn.functional.cross_entropy(logits, labels, weight=class_weights)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()

        # Validation
        model.eval()
        preds, gold = [], []
        with torch.no_grad():
            for batch in val_loader:
                input_ids = batch["input_ids"].squeeze(0).to(device)
                attn_mask = batch["attention_mask"].squeeze(0).to(device)
                labels = batch["label"].to(device)
                if labels.dim() == 0:
                    labels = labels.unsqueeze(0)
                logits = model(input_ids, attn_mask).unsqueeze(0)
                preds.append(torch.argmax(logits, dim=1).cpu().item())
                gold.append(labels.cpu().item())

        f1 = f1_score(gold, preds, average="micro")
        if f1 > best_f1:
            best_f1, best_state = f1, {k: v.cpu() for k,v in model.state_dict().items()}
            patience = 0
        else:
            patience += 1
            if patience >= EARLY_STOP:
                break

    return best_state, best_f1

In [ ]:
def evaluate_model(model, loader, device):
    model.eval()
    preds, gold = [], []
    with torch.no_grad():
        for batch in loader:
            input_ids = batch["input_ids"].squeeze(0).to(device)
            attn_mask = batch["attention_mask"].squeeze(0).to(device)
            labels = batch["label"].to(device)
            if labels.dim() == 0:
                labels = labels.unsqueeze(0)
            logits = model(input_ids, attn_mask).unsqueeze(0)
            preds.append(torch.argmax(logits, dim=1).cpu().item())
            gold.append(labels.cpu().item())

    report = classification_report(gold, preds, output_dict=True, zero_division=0)
    cm = confusion_matrix(gold, preds)
    return report, cm

In [ ]:
def run_experiment(encoder_name, encoder_model, spacy_model, exp_name):
    print(f"\n==== Running {exp_name} ====")
    nlp = spacy.load(spacy_model)
    tokenizer = AutoTokenizer.from_pretrained(encoder_model)

    df = pd.read_csv(DATA_CSV)[[ID_COL, TEXT_COL, LABEL_COL]].dropna()
    df[LABEL_COL] = df[LABEL_COL].astype(int)

    train_df, test_df = train_test_split(df, test_size=0.3, stratify=df[LABEL_COL], random_state=SEED)
    _, val_df = train_test_split(test_df, test_size=0.5, stratify=test_df[LABEL_COL], random_state=SEED)

    train_bags = split_and_enrich(train_df, nlp, TEXT_COL, ID_COL, LABEL_COL)
    val_bags   = split_and_enrich(val_df,   nlp, TEXT_COL, ID_COL, LABEL_COL)
    test_bags  = split_and_enrich(test_df,  nlp, TEXT_COL, ID_COL, LABEL_COL)

    train_ds, val_ds, test_ds = MILDataset(train_bags, tokenizer, MAX_LEN), MILDataset(val_bags, tokenizer, MAX_LEN), MILDataset(test_bags, tokenizer, MAX_LEN)
    train_loader, val_loader, test_loader = DataLoader(train_ds, BATCH_SIZE, shuffle=True), DataLoader(val_ds, BATCH_SIZE), DataLoader(test_ds, BATCH_SIZE)

    # ###################
    classes = np.unique(train_df[LABEL_COL])
    weights = compute_class_weight(class_weight='balanced', classes=classes, y=train_df[LABEL_COL])
    class_weights = torch.tensor(weights, dtype=torch.float).to(device)

    # ########### search best LR
    best_lr, best_val_f1, best_state = None, -1, None
    for lr in LEARNING_RATES:
        state, f1 = train_eval(train_loader, val_loader, device, class_weights, encoder_model, lr)
        if f1 > best_val_f1:
            best_val_f1, best_state, best_lr = f1, state, lr

    print(f"Best LR={best_lr}, Val F1={best_val_f1:.4f}")

    # ############## repeat experiment 5 times
    all_reports, all_cms = [], []
    for r in range(REPEATS):
        model = MILModel(encoder_model, freeze_encoder=True)
        model.load_state_dict(best_state)
        model.to(device)
        report, cm = evaluate_model(model, test_loader, device)
        report["repeat"] = r+1
        all_reports.append(report)
        all_cms.append(cm)

    # Save
    flat_reports = []
    for r, report in enumerate(all_reports):
        row = {"repeat": r+1}
        for label, metrics in report.items():
            if isinstance(metrics, dict):
                for k,v in metrics.items():
                    row[f"{label}_{k}"] = v
            else:
                row[label] = metrics
        flat_reports.append(row)

    avg_report = pd.DataFrame(flat_reports).mean(numeric_only=True).to_dict()
    avg_report["repeat"] = "avg"
    flat_reports.append(avg_report)

    out_csv = os.path.join(OUTPUT_DIR, f"{exp_name}.csv")
    pd.DataFrame(flat_reports).to_csv(out_csv, index=False)
    print(f"Saved results to {out_csv}")

In [ ]:
if __name__ == "__main__":
    for enc_name, enc_model in ENCODER_MODELS.items():
        for spacy_key, spacy_model in SCISPACY_MODELS.items():
            exp_name = f"{enc_name}_{spacy_key}"
            run_experiment(enc_name, enc_model, spacy_model, exp_name)